In [25]:
import json
import time
import syncedlyrics
from mutagen.flac import FLAC
from mutagen.mp4 import MP4
from pathlib import Path
from tqdm import tqdm

MP4_TAG_MAP = {"title": "©nam", "artist": "©ART", "album": "©alb"}

def get_flac_metadata(filepath: str) -> dict:
    audio = FLAC(filepath)
    return {k: (audio.get(k) or [""])[0].strip() for k in ("title", "artist", "album")}

def get_alac_metadata(filepath: str) -> dict:
    audio = MP4(filepath)
    return {
        k: (audio.tags.get(tag) or [""])[0].strip()
        for k, tag in MP4_TAG_MAP.items()
    }

def get_metadata(filepath: Path) -> dict | None:
    try:
        if filepath.suffix.lower() == ".flac":
            return get_flac_metadata(str(filepath))
        elif filepath.suffix.lower() == ".m4a":
            return get_alac_metadata(str(filepath))
    except Exception as e:
        print(f"  Ошибка метаданных {filepath.name}: {e}")
    return None

def get_lyrics(title: str, artist: str) -> str | None:
    try:
        return syncedlyrics.search(f"{title} {artist}", plain_only = True)
    except Exception:
        return None

def fetch_lyrics_bulk(folder: str, output_file: str = "lyrics.json", delay: float = 0.5):
    results = {}
    audio_files = [
        p for p in Path(folder).rglob("*")
        if p.suffix.lower() in (".flac", ".m4a")
    ]
    print(f"Найдено файлов: {len(audio_files)}")

    for i, filepath in tqdm(enumerate(audio_files, 1), total = len(audio_files)):
        meta = get_metadata(filepath)
        if not meta or not meta["title"] or not meta["artist"]:
            print(f"[{i}/{len(audio_files)}] Пропуск: {filepath.name}")
            continue

        key = f"{meta['artist']} — {meta['title']}"
        if key in results:
            continue

        lyrics = get_lyrics(meta["title"], meta["artist"])
        status = "✓" if lyrics else "✗"
        print(f"[{i}/{len(audio_files)}] {status} {key}")

        results[key] = {**meta, "lyrics": lyrics}
        time.sleep(delay)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    found = sum(1 for v in results.values() if v["lyrics"])
    print(f"\nГотово: {found}/{len(results)} текстов найдено → {output_file}")
    return results


In [26]:
fetch_lyrics_bulk(
    folder="S:\Music\Glass Animals - Dreamland (2020)")

<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:2: SyntaxWarning: invalid escape sequence '\M'
C:\Users\ivans\AppData\Local\Temp\ipykernel_45760\2471499744.py:2: SyntaxWarning: invalid escape sequence '\M'
  folder="S:\Music\Glass Animals - Dreamland (2020)")


Найдено файлов: 16


0it [00:00, ?it/s]

[1/16] ✓ Glass Animals — Dreamland


1it [00:02,  2.08s/it]

...

2it [00:03,  1.42s/it]

[3/16] ✓ Glass Animals — ((home movie: 1994))


3it [00:04,  1.24s/it]

[4/16] ✓ Glass Animals — Hot Sugar


4it [00:09,  2.29s/it]
Exception ignored in: <finalize object at 0x2ab67c08ba0; dead>
Traceback (most recent call last):
  File "C:\Users\ivans\anaconda3\Lib\weakref.py", line 588, in __call__
KeyboardInterrupt: 

KeyboardInterrupt



In [19]:
with open('lyrics.json', 'r', encoding = 'utf-8') as f:
    data = json.load(f)

In [22]:
[(k, v.get('lyrics').splitlines()) for k, v in data.items()][2]

('Glass Animals — ((home movie: 1994))',
 ['What is the date today?', '27th of May, 1994'])